# Support Integrity Auditor (SIA) — Reproducible Pipeline

This notebook is the end-to-end, reproducible pipeline mandated by the project
spec (`proj_details.pdf`, §7):

> **`notebook.ipynb`** — Full reproducible pipeline (pseudo-labeling → training → inference).

It stitches together the project's standalone scripts:

| Stage | Script | What it does |
|---|---|---|
| **1 — Pseudo-Label Generation** (self-supervised) | `script/pseudo_label_generation.py` | Fuses ≥2 independent signals into an inferred severity, then derives the binary `Is_Mismatch` label by comparing it to the human-assigned `Priority_Level`. |
| **2 — Classifier Training** (fine-tuned heads) | `script/train_classifier.py` | Trains the `SeverityClassifier` (MiniLM text embeddings + structured metadata) on the pseudo-labeled data. |
| **3 — Evidence Dossier Generation** | `script/evidence_dossier_generation.py` | For every flagged mismatch, emits a structured, hallucination-free Evidence Dossier. |

**Architecture (pipeline diagram):**

```
 raw tickets ─┐
              ├─► [Stage 1] LLM score (0.30) ┐
              │            embedding score (0.35) ├─► fused severity ─► compare to Priority ─► Is_Mismatch
              └─►          cat/resolution (0.35) ┘                                                 │
                                                                                                   ▼
                          [Stage 2] MiniLM emb + metadata ──► SeverityClassifier ──► inferred severity
                                                                                                   │
                                                                                                   ▼
                          [Stage 3] inferred vs assigned ──► EvidenceDossierGenerator ──► dossier JSON
```


## 0. Setup

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd

ROOT = os.path.abspath(".")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

pd.set_option("display.max_colwidth", 80)
DATA = "./dataset/preprocessed_data.csv"
MODEL_PATH = "./Artifacts/severity_classifier.joblib"

## Stage 1 — Pseudo-Label Generation (self-supervised)

The spec requires fusing **at least two independent signals** into an inferred
severity, *independent of the `Ticket_Priority` column*. `pseudo_label_generation.py`
fuses **three**:

| Signal | Weight | Source field(s) |
|---|---|---|
| LLM zero-shot severity (Gemma-2-2B-it, 4-bit) | 0.30 | `Ticket_Subject`, `Ticket_Description` |
| Embedding similarity to severity anchors (MiniLM) | 0.35 | `Ticket_Subject`, `Ticket_Description` |
| Category + resolution-time lookup | 0.35 | `Issue_Category`, `Resolution_Time_Hours` |

```python
final_scores[label] = 0.30*llm[label] + 0.35*embedding[label] + 0.35*cat_res[label]
Final_severity_label = argmax(final_scores)
Is_Mismatch          = int(Priority_Level.lower() != Final_severity_label.lower())
```

> ⚠️ **Regenerating the labels from scratch is GPU-heavy** (it loads a 2B-param LLM).
> The cell below shows *how* to do it but is disabled by default. We then load the
> already-generated `preprocessed_data.csv` for the rest of the pipeline.


In [ ]:
REGENERATE_PSEUDO_LABELS = False

if REGENERATE_PSEUDO_LABELS:
    from script.pseudo_label_generation import batch_classify_tickets, is_mismatch
    raw = pd.read_csv("./dataset/enhanced_customer_support_data.csv")
    raw = batch_classify_tickets(raw)
    raw["Is_Mismatch"] = raw.apply(is_mismatch, axis=1)
    raw.to_csv(DATA, index=False)
    print("Regenerated", DATA)
else:
    print("Skipping regeneration — using the precomputed", DATA)

In [ ]:
df = pd.read_csv(DATA)
print(f"Loaded {len(df):,} pseudo-labeled tickets")
print("\nInferred severity distribution:")
print(df["Final_severity_label"].value_counts())
print("\nIs_Mismatch distribution (1 = priority mismatch):")
print(df["Is_Mismatch"].value_counts())
print(f"\nMismatch rate: {df['Is_Mismatch'].mean():.1%}")
df[["Ticket_ID", "Priority_Level", "Final_severity_label", "Is_Mismatch"]].head()

### Stage-1 evaluation metric: Pairwise Signal Agreement (§5)

The spec asks for the **pairwise agreement between the two chosen signals**. Below
we recompute the two CPU-cheap signals — the **MiniLM embedding** signal and the
**category/resolution** signal — on a sample and measure how often their `argmax`
severities agree. (These reuse the exact anchor sentences and lookup table from
`pseudo_label_generation.py`; the LLM signal is omitted here only because it needs
a GPU.)


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

SCORING = {
    "LOW": ["for new features", "is not updating", "not receiving the password reset email", "How do i"],
    "MEDIUM": ["application crashes", "data hasn't synced", "dashboard is not loading any data"],
    "HIGH": ["cannot log into my account even after resetting the password", "cannot pass the 2FA check",
             "trying to update payment method", "payment processing issue"],
    "CRITICAL": ["received a login alert", "Lock my account immediately", "Someone used my",
                 "receiving a 500 Internal Server Error"],
}
LABELS = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]

_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
_anchor_emb = {k: _encoder.encode(v) for k, v in SCORING.items()}

def embedding_signal(text):
    e = _encoder.encode(text)
    return {k: float(np.max(cosine_similarity([e], a)[0])) for k, a in _anchor_emb.items()}

_CAT = {
    "General Inquiry": (0.061, 0.112, 0.144, 0.683), "Technical": (0.142, 0.212, 0.665, 0.146),
    "Account": (0.120, 0.228, 0.149, 0.503), "Billing": (0.051, 0.510, 0.224, 0.215),
    "Fraud": (0.459, 0.261, 0.023, 0.257),
}
def cat_res_signal(cat, hrs):
    cc, ch, cm, cl = _CAT.get(cat, (0.10, 0.25, 0.35, 0.30))
    if hrs < 24:      tc, th, tm, tl = 0.317, 0.278, 0.250, 0.221
    elif hrs < 48:    tc, th, tm, tl = 0.219, 0.261, 0.256, 0.264
    elif hrs < 72:    tc, th, tm, tl = 0.266, 0.0,   0.336, 0.397
    else:             tc, th, tm, tl = 0.111, 0.357, 0.235, 0.301
    crit, high, med, low = (cc+tc)/2, (ch+th)/2, (cm+tm)/2, (cl+tl)/2
    tot = crit + high + med + low
    return {"CRITICAL": crit/tot, "HIGH": high/tot, "MEDIUM": med/tot, "LOW": low/tot}

sample = df.sample(min(500, len(df)), random_state=42)
emb_lab, cat_lab = [], []
for _, r in sample.iterrows():
    es = embedding_signal(f"{r.Ticket_Subject} {r.Ticket_Description}")
    cs = cat_res_signal(r.Issue_Category, r.Resolution_Time_Hours)
    emb_lab.append(max(es, key=es.get))
    cat_lab.append(max(cs, key=cs.get))

agreement = float(np.mean([a == b for a, b in zip(emb_lab, cat_lab)]))
print(f"Pairwise Signal Agreement (embedding vs category/resolution, n={len(sample)}): {agreement:.3f}")

## Stage 2 — Classifier Training (fine-tuned model)

`SeverityClassifier` (in `train_classifier.py`) is a two-head fusion model:

- **Text head** — 384-d MiniLM (`all-MiniLM-L6-v2`) sentence embedding of
  `subject + description`, fed to an XGBoost classifier.
- **Metadata head** — one-hot `Issue_Category` + `Ticket_Channel` and scaled
  `Resolution_Time_Hours` + `Satisfaction_Score`, fed to a second XGBoost head.

Final severity = `0.6 * text_proba + 0.4 * meta_proba` (argmax). This satisfies the
spec's *"text fields **and** ≥1 structured metadata feature"* requirement.

**Why predict severity then compare?** As documented in `STRATEGIES.md`, predicting
`Is_Mismatch` *directly* tops out near ~72% because it forces the model to learn a
priority×severity interaction. **Decomposing** the task — predict the 4-class
severity, then deterministically set `Is_Mismatch = (Priority != predicted_severity)`
— clears all three §6 thresholds with large margin.


In [ ]:
from script.train_classifier import SeverityClassifier


TRAIN_SAMPLE = None   

clf = SeverityClassifier(weights=(0.6, 0.4))
data = clf.load_data(DATA)
if TRAIN_SAMPLE:
    data = data.sample(TRAIN_SAMPLE, random_state=42)

report = clf.fit(data)
print(report)

In [ ]:
clf.save(MODEL_PATH)

## Stage 3 — Inference & Evidence Dossier Generation

We load the trained classifier, infer severity on unseen tickets, derive the binary
mismatch judgment, and emit a grounded **Evidence Dossier** for every flagged ticket
via `EvidenceDossierGenerator`. Every `feature_evidence` item is traceable to a real
input field — **zero hallucination** (the Hard Rule in §4 Stage 3).


In [ ]:
from script.train_classifier import SeverityClassifier
from script.evidence_dossier_generation import EvidenceDossierGenerator

SEVERITY_LABELS = {0: "LOW", 1: "MEDIUM", 2: "HIGH", 3: "CRITICAL"}
clf = SeverityClassifier.load(MODEL_PATH)
gen = EvidenceDossierGenerator()

def predict_severity_proba(clf, frame):
    """Fused 4-class severity probabilities (mirrors SeverityClassifier.predict)."""
    embd_X = clf._build_embeddings(frame)
    meta_X = clf._build_meta(frame)
    meta_X[clf.SCALE_COLS] = clf.scaler.transform(meta_X[clf.SCALE_COLS])
    meta_X = clf._align_meta_columns(meta_X)
    embd_prob = clf.embd_model.predict_proba(embd_X)
    meta_prob = clf.meta_model.predict_proba(meta_X)
    return clf.weights[0] * embd_prob + clf.weights[1] * meta_prob

def audit(frame):
    frame = frame.copy().reset_index(drop=True)
    proba = predict_severity_proba(clf, frame)
    frame["Inferred_Severity"] = [SEVERITY_LABELS[i] for i in np.argmax(proba, axis=1)]
    frame["Is_Mismatch"] = (frame["Priority_Level"].str.upper()
                            != frame["Inferred_Severity"].str.upper()).astype(int)
    dossiers = []
    for i, row in frame.iterrows():
        scores = {SEVERITY_LABELS[j]: float(proba[i, j]) for j in range(4)}
        dossiers.append(gen.generate(row, inferred_severity=row["Inferred_Severity"], scores=scores))
    return frame, dossiers

In [ ]:
sample = pd.read_csv(DATA).sample(20, random_state=7)
result, dossiers = audit(sample)
result[["Ticket_ID", "Priority_Level", "Inferred_Severity", "Is_Mismatch"]].head(20)

In [ ]:
for d in [x for x in dossiers if x is not None][:3]:
    print(gen.to_json(d))
    print("-" * 70)